# 🔧 Churn Prediction - Feature Engineering

**Goal:** Merge all data sources into a single feature matrix for model training.

**Feature categories:**
- Customer demographics (industry, country, company_size)
- Subscription features (plan, price, renewal days)
- Subscription events (upgrades, downgrades)
- Support tickets (count, priority, resolution time)
- Activity metrics (login count, session duration, engagement, feature usage)

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


In [2]:
# Load all datasets
customers = pd.read_csv('data/customers.csv')
subscriptions = pd.read_csv('data/subscriptions.csv')
sub_events = pd.read_csv('data/subscription_events.csv')
tickets = pd.read_csv('data/support_tickets.csv')
activity = pd.read_csv('data/activity_metrics.csv')

print('Data loaded:')
for df, name in [(customers,'customers'),(subscriptions,'subscriptions'),(sub_events,'sub_events'),(tickets,'tickets'),(activity,'activity')]:
    print(f'  {name}: {df.shape}')

Data loaded:
  customers: (2000, 11)
  subscriptions: (2000, 8)
  sub_events: (2000, 6)
  tickets: (1027, 9)
  activity: (2000, 9)


## 1. Create Target Variable

In [3]:
# Target: 1 if churned (inactive), 0 if active
customers['is_churned'] = (customers['status'] == 'inactive').astype(int)

print(f'Target distribution:')
print(f'  Active: {(customers["is_churned"] == 0).sum()}')
print(f'  Churned: {(customers["is_churned"] == 1).sum()}')
print(f'  Churn rate: {customers["is_churned"].mean()*100:.2f}%')

Target distribution:
  Active: 1738
  Churned: 262
  Churn rate: 13.10%


## 2. Customer Demographics Features

In [4]:
# Encode categorical columns
categorical_cols = ['industry', 'country']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    customers[col + '_encoded'] = le.fit_transform(customers[col].fillna('Unknown'))
    label_encoders[col] = le
    print(f'{col}: {len(le.classes_)} unique values -> encoded')

# Company size - log transform to normalize
customers['company_size_log'] = np.log1p(customers['company_size'].fillna(0))

# Registration tenure in days
customers['created_at'] = pd.to_datetime(customers['created_at'])
customers['tenure_days'] = (pd.Timestamp.now() - customers['created_at']).dt.days.astype(int)

print(f'Tenure days range: {customers["tenure_days"].min()} - {customers["tenure_days"].max()}')

industry: 5 unique values -> encoded
country: 241 unique values -> encoded
Tenure days range: 0 - 0


## 3. Subscription Features

In [5]:
# Merge subscription (one per customer - left join)
cust = customers.copy()

# Get latest subscription per customer
subscriptions['renewal_date'] = pd.to_datetime(subscriptions['renewal_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
sub_latest = subscriptions.sort_values('renewal_date', ascending=False).drop_duplicates('customer_id')

# Merge using customers.id -> subscriptions.customer_id
cust = cust.merge(
    sub_latest[['customer_id', 'plan_name', 'monthly_price', 'start_date', 'renewal_date']],
    left_on='id', right_on='customer_id', how='left'
)
cust = cust.drop(columns=['customer_id_y'], errors='ignore')
cust = cust.rename(columns={'customer_id_x': 'customer_id'})

print(f'After subscription merge: {cust.shape}')
print(f'Missing plans: {cust["plan_name"].isna().sum()}')

After subscription merge: (2000, 21)
Missing plans: 0


In [6]:
# Encode plan name
le_plan = LabelEncoder()
cust['plan_name_encoded'] = le_plan.fit_transform(cust['plan_name'].fillna('Unknown').astype(str))
label_encoders['plan_name'] = le_plan
print(f'Plans: {list(le_plan.classes_)}')

# Monthly price - fill NaN with median
cust['monthly_price'] = cust['monthly_price'].fillna(cust['monthly_price'].median())

# Days to renewal
cust['days_to_renewal'] = (cust['renewal_date'] - pd.Timestamp.now()).dt.days.fillna(-1).astype(int)
cust['days_since_start'] = (pd.Timestamp.now() - cust['start_date']).dt.days.fillna(-1).astype(int)

# Subscription length
cust['subscription_length_days'] = (cust['renewal_date'] - cust['start_date']).dt.days
median_length = cust['subscription_length_days'].median()
cust['subscription_length_days'] = cust['subscription_length_days'].fillna(median_length).astype(int)

Plans: ['Business', 'Enterprise', 'Growth', 'Starter']


## 4. Subscription Events Features

In [7]:
# Aggregate subscription events
event_features = sub_events.groupby('customer_id').agg(
    total_events=('event_type', 'count'),
    n_upgrades=('event_type', lambda x: (x == 'upgrade').sum()),
    n_downgrades=('event_type', lambda x: (x == 'downgrade').sum()),
    n_cancellations=('event_type', lambda x: (x == 'cancellation').sum())
).reset_index()

cust = cust.merge(event_features, on='customer_id', how='left')

# Fill NaN with 0 (customers with no events)
event_cols = ['total_events', 'n_upgrades', 'n_downgrades', 'n_cancellations']
for col in event_cols:
    cust[col] = cust[col].fillna(0).astype(int)

# Ratio features
cust['upgrade_ratio'] = cust['n_upgrades'] / (cust['total_events'] + 1)
cust['downgrade_ratio'] = cust['n_downgrades'] / (cust['total_events'] + 1)
cust['has_downgraded'] = (cust['n_downgrades'] > 0).astype(int)

print(f'Event features added. Customers with events: {(cust["total_events"] > 0).sum()}')

Event features added. Customers with events: 2000


## 5. Support Ticket Features

In [8]:
# Parse dates
tickets['created_at'] = pd.to_datetime(tickets['created_at'])
tickets['resolved_at'] = pd.to_datetime(tickets['resolved_at'])

# Resolution time in hours
tickets['resolution_hours'] = (tickets['resolved_at'] - tickets['created_at']).dt.total_seconds() / 3600

# Aggregate ticket features
ticket_features = tickets.groupby('customer_id').agg(
    ticket_count=('id', 'count'),
    high_priority=('priority', lambda x: (x == 'High').sum()),
    urgent_priority=('priority', lambda x: (x == 'Urgent').sum()),
    open_tickets=('status', lambda x: (x != 'Closed').sum()),
    closed_tickets=('status', lambda x: (x == 'Closed').sum()),
    tech_issues=('category', lambda x: (x == 'Technical Issue').sum()),
    billing_issues=('category', lambda x: (x == 'Billing').sum()),
    avg_resolution_hours=('resolution_hours', 'mean'),
    max_resolution_hours=('resolution_hours', 'max')
).reset_index()

cust = cust.merge(ticket_features, on='customer_id', how='left')

# Fill NaN for customers with no tickets
ticket_cols = ['ticket_count', 'high_priority', 'urgent_priority', 'open_tickets', 
               'closed_tickets', 'tech_issues', 'billing_issues', 
               'avg_resolution_hours', 'max_resolution_hours']
for col in ticket_cols:
    cust[col] = cust[col].fillna(0)

# Ratio features
cust['high_priority_ratio'] = cust['high_priority'] / (cust['ticket_count'] + 1)
cust['tech_issue_ratio'] = cust['tech_issues'] / (cust['ticket_count'] + 1)
cust['billing_issue_ratio'] = cust['billing_issues'] / (cust['ticket_count'] + 1)
cust['has_open_tickets'] = (cust['open_tickets'] > 0).astype(int)

print(f'Ticket features added. Customers with tickets: {(cust["ticket_count"] > 0).sum()}')

Ticket features added. Customers with tickets: 1027


## 6. Activity Metrics Features

In [9]:
# Parse dates
activity['metric_date'] = pd.to_datetime(activity['metric_date'])
activity['last_active_date'] = activity.groupby('customer_id')['metric_date'].transform('max')
activity['days_since_last_active'] = (pd.Timestamp.now() - activity['last_active_date']).dt.days

# Aggregate activity
activity_features = activity.groupby('customer_id').agg(
    avg_login_count=('login_count', 'mean'),
    total_login_count=('login_count', 'sum'),
    max_login_count=('login_count', 'max'),
    avg_session_duration=('session_duration', 'mean'),
    total_session_duration=('session_duration', 'sum'),
    max_session_duration=('session_duration', 'max'),
    avg_feature_usage=('feature_usage_score', 'mean'),
    max_feature_usage=('feature_usage_score', 'max'),
    min_feature_usage=('feature_usage_score', 'min'),
    avg_engagement=('engagement_score', 'mean'),
    max_engagement=('engagement_score', 'max'),
    min_engagement=('engagement_score', 'min'),
    avg_active_days=('active_days', 'mean'),
    total_active_days=('active_days', 'sum'),
    max_active_days=('active_days', 'max'),
    metric_count=('metric_date', 'count'),
    days_since_last_active=('days_since_last_active', 'min'),
    login_trend=('login_count', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0),
    engagement_trend=('engagement_score', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0),
    feature_trend=('feature_usage_score', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0),
    session_trend=('session_duration', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0)
).reset_index()

cust = cust.merge(activity_features, on='customer_id', how='left')

# Fill NaN for customers with no activity
activity_cols_list = [c for c in activity_features.columns if c != 'customer_id']
for col in activity_cols_list:
    cust[col] = cust[col].fillna(0)

print(f'Activity features added. Rows: {len(cust)}')
print(f'Columns: {cust.shape[1]}')

Activity features added. Rows: 2000
Columns: 66


## 7. Final Feature Selection

In [10]:
# Select features for model training
feature_cols = [
    # Demographics
    'industry_encoded', 'country_encoded', 'company_size_log', 'tenure_days',
    # Subscription
    'plan_name_encoded', 'monthly_price', 'days_to_renewal', 'days_since_start',
    'subscription_length_days',
    # Subscription events
    'total_events', 'n_upgrades', 'n_downgrades', 'n_cancellations',
    'upgrade_ratio', 'downgrade_ratio', 'has_downgraded',
    # Support tickets
    'ticket_count', 'high_priority', 'urgent_priority', 'open_tickets', 'closed_tickets',
    'tech_issues', 'billing_issues', 'avg_resolution_hours', 'max_resolution_hours',
    'high_priority_ratio', 'tech_issue_ratio', 'billing_issue_ratio', 'has_open_tickets',
    # Activity
    'avg_login_count', 'total_login_count', 'max_login_count',
    'avg_session_duration', 'total_session_duration', 'max_session_duration',
    'avg_feature_usage', 'max_feature_usage', 'min_feature_usage',
    'avg_engagement', 'max_engagement', 'min_engagement',
    'avg_active_days', 'total_active_days', 'max_active_days',
    'metric_count', 'days_since_last_active',
    'login_trend', 'engagement_trend', 'feature_trend', 'session_trend'
]

print(f'Total features selected: {len(feature_cols)}')

Total features selected: 50


In [11]:
# Prepare X and y
X = cust[feature_cols].copy()
y = cust['is_churned'].copy()

# Handle any remaining NaN values
for col in X.columns:
    if X[col].isna().sum() > 0:
        X[col] = X[col].fillna(X[col].median())

# Scale numerical features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print(f'Final feature matrix: {X_scaled.shape}')
print(f'Target: {y.value_counts().to_dict()}')

Final feature matrix: (2000, 50)
Target: {0: 1738, 1: 262}


In [12]:
# Save preprocessor objects
preprocess = {
    'label_encoders': label_encoders,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'categorical_cols': categorical_cols
}

joblib.dump(preprocess, '../backend/trained_models/preprocess.pkl')
print('✅ preprocess.pkl saved to backend/trained_models/')

✅ preprocess.pkl saved to backend/trained_models/


In [13]:
# Save the processed dataset for model training
X_scaled.to_pickle('data/X_features.pkl')
y.to_pickle('data/y_target.pkl')
cust[['id', 'customer_id', 'is_churned'] + feature_cols].to_csv('data/master_dataset.csv', index=False)

print('✅ Processed dataset saved')
print('Feature Engineering Complete! Ready for Model Training.')

✅ Processed dataset saved
Feature Engineering Complete! Ready for Model Training.
